# Bi-LSTM + Transformer Glucose Predictor — V2 (5-min horizon, Clean Data)
**Matches LSTM+Transformer approach**: 5-min ahead, sliding window, chronological per-patient split, no leakage.
**Added**: Patient filtering, flat-line removal, glucose capping for clean signals.

**Upload to Colab** → Runtime → Change runtime type → GPU (T4)
**Upload `subdatabaseII.csv`** to Colab (folder icon → upload) before running

In [ ]:
!pip install -q tqdm scikit-learn matplotlib pandas numpy torch

In [ ]:
import os, random, gc, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# CONFIG — Matches LSTM+Transformer exactly
# ============================================================
CSV_PATH = "subdatabaseII.csv"
WINDOW_SIZE = 12          # 60 min history (12 × 5 min)
HORIZON = 1               # 5 min ahead (next step) — KEY: matches lstmtransformer.py
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
WD = 1e-5                 # Matches lstmtransformer.py weight_decay
HIDDEN = 64
HEADS = 4
LAYERS = 2
DROPOUT = 0.1
PATIENCE = 5
MIN_DELTA = 1e-4
MAX_GAP_MIN = 5
RANDOM_STATE = 42
USE_AMP = True

# Data cleaning thresholds
MAX_MISSING_PCT = 0.30    # Drop patients with >30% missing glucose
MAX_FLAT_RUN = 20         # Remove runs of >20 identical consecutive values
GLUCOSE_CAP = 400.0       # Cap at physiological max (mg/dL)
MIN_PATIENT_ROWS = 500    # Minimum clean rows per patient

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_STATE)

In [ ]:
# ============================================================
# LOAD & CLEAN — With patient filtering & artifact removal
# ============================================================
df = pd.read_csv(CSV_PATH)
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Unique patients: {df['PtID'].nunique()}")

# Drop extra column if present
if 'Database' in df.columns:
    df = df.drop(columns=['Database'])

req = ["ts", "PtID", "GlucoseCGM", "HR"]
for c in req:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}. Found: {list(df.columns)}")

df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["GlucoseCGM"] = pd.to_numeric(df["GlucoseCGM"], errors="coerce")
df["HR"] = pd.to_numeric(df["HR"], errors="coerce")

df = df.dropna(subset=["ts", "PtID"]).copy()
df = df.sort_values(["PtID", "ts"]).reset_index(drop=True)

# ============================================================
# PATIENT-LEVEL CLEANING
# ============================================================
clean_patients = []
patient_stats = []

for pid, pdf in df.groupby("PtID", sort=False):
    pdf = pdf.sort_values("ts").reset_index(drop=True)
    
    total_rows = len(pdf)
    missing_pct = pdf["GlucoseCGM"].isna().mean()
    
    # Skip patients with too much missing data
    if missing_pct > MAX_MISSING_PCT:
        patient_stats.append((pid, total_rows, missing_pct, "DROPPED: high missing"))
        continue
    
    # Cap glucose at physiological max
    pdf.loc[pdf["GlucoseCGM"] > GLUCOSE_CAP, "GlucoseCGM"] = GLUCOSE_CAP
    
    clean_rows = pdf["GlucoseCGM"].notna().sum()
    
    if clean_rows >= MIN_PATIENT_ROWS:
        clean_patients.append(pdf)
        patient_stats.append((pid, total_rows, missing_pct, f"KEPT: {clean_rows} clean rows"))
    else:
        patient_stats.append((pid, total_rows, missing_pct, "DROPPED: too few clean rows"))

print(f"\nPatient filtering results:")
for pid, total, missing, status in patient_stats:
    print(f"  {pid:25s} | {total:6d} rows | {missing*100:5.1f}% missing | {status}")

df = pd.concat(clean_patients, ignore_index=True) if clean_patients else pd.DataFrame()
print(f"\nAfter cleaning: {df.shape[0]} rows, {df['PtID'].nunique()} patients")

# Time features — circadian encoding
mins = df["ts"].dt.hour * 60 + df["ts"].dt.minute
df["time_sin"] = np.sin(2 * np.pi * mins / 1440)
df["time_cos"] = np.cos(2 * np.pi * mins / 1440)

FEATURES = ["GlucoseCGM", "HR", "time_sin", "time_cos"]
print(f"Features: {FEATURES}")
print(f"Missing glucose: {df['GlucoseCGM'].isna().sum()}, HR: {df['HR'].isna().sum()}")

In [ ]:
# ============================================================
# CHRONOLOGICAL PER-PATIENT SPLIT — NO LEAKAGE
# Exact same logic as lstmtransformer.py split_data()
# ============================================================
splits = {"train": [], "val": [], "test": []}

for patient_id, patient_data in df.groupby("PtID", sort=False):
    patient_data = patient_data.sort_values("ts").reset_index(drop=True)
    n = len(patient_data)
    
    if n < WINDOW_SIZE + 3:  # Need at least window + target + buffer
        continue

    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    if train_end >= WINDOW_SIZE + 1:
        splits["train"].append(patient_data.iloc[:train_end].copy())
    if val_end - train_end >= WINDOW_SIZE + 1:
        splits["val"].append(patient_data.iloc[train_end:val_end].copy())
    if n - val_end >= WINDOW_SIZE + 1:
        splits["test"].append(patient_data.iloc[val_end:].copy())

if not splits["train"]:
    raise ValueError("Not enough data to create training sequences.")

# HR imputation — median from TRAIN ONLY (no leakage)
training_data = pd.concat(splits["train"], ignore_index=True)
hr_median = training_data["HR"].median()
if pd.isna(hr_median):
    hr_median = 0.0

for split_name in splits:
    for i in range(len(splits[split_name])):
        group = splits[split_name][i].copy()
        group["HR"] = group["HR"].fillna(hr_median)
        splits[split_name][i] = group

for name in splits:
    row_count = sum(len(g) for g in splits[name])
    print(f"{name}: {row_count} rows")

In [ ]:
# ============================================================
# CREATE SEQUENCES — Sliding window, exact match to lstmtransformer.py
# ============================================================
def create_sequences(groups, feature_scaler, target_scaler, fit_scalers=False):
    X_list, y_list = [], []
    max_gap = pd.Timedelta(minutes=MAX_GAP_MIN)

    if fit_scalers:
        train_data = pd.concat(groups, ignore_index=True)
        valid = train_data["GlucoseCGM"].notna()
        feature_scaler.fit(train_data.loc[valid, FEATURES].to_numpy(np.float32))
        target_scaler.fit(train_data.loc[valid, ["GlucoseCGM"]].to_numpy(np.float32))

    for g in groups:
        g = g.sort_values("ts").reset_index(drop=True)
        vals = g[FEATURES].to_numpy(np.float32)
        gluc_valid = g["GlucoseCGM"].notna().to_numpy()
        times = g["ts"].tolist()

        # Sliding window: end goes from WINDOW_SIZE to len(g)
        # Target is at 'end' (next step = 5 min ahead)
        for end in range(WINDOW_SIZE, len(g)):
            start = end - WINDOW_SIZE

            # All input window + target must have valid glucose
            if not gluc_valid[start:end + 1].all():
                continue

            # Check time gaps ≤ 5 min
            window_times = times[start:end + 1]
            gaps = [window_times[i+1] - window_times[i] for i in range(len(window_times)-1)]
            if not all(pd.Timedelta(0) < gap <= max_gap for gap in gaps):
                continue

            input_sequence = vals[start:end]      # 12 steps
            target_glucose = vals[end, 0]         # Next glucose

            X_scaled = feature_scaler.transform(input_sequence).astype(np.float32)
            y_scaled = target_scaler.transform(np.array([[target_glucose]], np.float32)).astype(np.float32)[0]

            X_list.append(X_scaled)
            y_list.append(y_scaled)

    if not X_list:
        return np.empty((0, WINDOW_SIZE, len(FEATURES)), np.float32), np.empty((0,), np.float32)
    return np.stack(X_list), np.stack(y_list)

fs, ts = StandardScaler(), StandardScaler()
X_train, y_train = create_sequences(splits["train"], fs, ts, fit_scalers=True)
X_val, y_val = create_sequences(splits["val"], fs, ts)
X_test, y_test = create_sequences(splits["test"], fs, ts)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
    raise ValueError("Empty split — check MAX_GAP_MIN or data quality")

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).view(-1,1))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val).view(-1,1))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test).view(-1,1))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# ============================================================
# MODEL: Bi-LSTM + Transformer (V2 Architecture)
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class BiLSTMTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(len(FEATURES), HIDDEN)
        self.pos = PositionalEncoding(HIDDEN)
        
        # Bi-LSTM branch — bidirectional captures past context better
        self.bilstm = nn.LSTM(HIDDEN, HIDDEN, num_layers=1, batch_first=True,
                            bidirectional=True, dropout=0.0)
        
        # Transformer branch
        enc = nn.TransformerEncoderLayer(HIDDEN, HEADS, HIDDEN*4, DROPOUT,
                                       batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc, LAYERS)
        
        # Fusion: Bi-LSTM gives 2*HIDDEN, Transformer gives HIDDEN
        self.fusion = nn.Sequential(
            nn.Linear(3*HIDDEN, HIDDEN), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN, HIDDEN//2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN//2, 1)
        )
    
    def forward(self, x):
        x = self.proj(x)
        x = self.pos(x)
        
        lstm_out, _ = self.bilstm(x)          # (B, T, 2*H)
        lstm_feat = lstm_out[:, -1, :]        # Last timestep
        
        trans_out = self.transformer(x)       # (B, T, H)
        trans_feat = trans_out[:, -1, :]      # Last timestep
        
        combined = torch.cat([lstm_feat, trans_feat], dim=1)  # (B, 3*H)
        return self.fusion(combined)

In [ ]:
# ============================================================
# TRAINING SETUP — Matches lstmtransformer.py
# ============================================================
model = BiLSTMTransformer().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=2)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type=='cuda' and USE_AMP else None

def train_epoch(model, loader, opt, crit, scaler):
    model.train()
    total = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        if scaler:
            with torch.autocast('cuda', dtype=torch.float16):
                pred = model(X)
                loss = crit(pred, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            pred = model(X)
            loss = crit(pred, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        total += loss.item() * len(X)
    return total / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval()
    total, preds, actuals = 0.0, [], []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        loss = crit(pred, y)
        total += loss.item() * len(X)
        preds.append(pred.cpu().numpy())
        actuals.append(y.cpu().numpy())
    return total / len(loader.dataset), np.concatenate(preds), np.concatenate(actuals)

In [ ]:
# ============================================================
# TRAINING LOOP WITH EARLY STOPPING
# ============================================================
best_loss = float('inf')
best_state = None
no_improve = 0
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    
    improved = val_loss < best_loss - MIN_DELTA
    if improved:
        best_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        # Save checkpoint — matches lstmtransformer.py format
        torch.save({
            'model_state_dict': best_state,
            'window_size': WINDOW_SIZE,
            'hidden_size': HIDDEN,
            'num_heads': HEADS,
            'transformer_layers': LAYERS,
            'feature_columns': FEATURES,
            'hr_median': hr_median,
            'feature_mean': fs.mean_.tolist(),
            'feature_scale': fs.scale_.tolist(),
            'target_mean': ts.mean_.tolist(),
            'target_scale': ts.scale_.tolist()
        }, 'bilstm_transformer_5min.pth')
    else:
        no_improve += 1
    
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {tr_loss:.6f} | Val: {val_loss:.6f} {'✓' if improved else ''}")
    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
print(f"Best val loss: {best_loss:.6f}")

In [ ]:
# ============================================================
# TEST EVALUATION + SAVE OUTPUTS
# ============================================================
test_loss, pred_scaled, actual_scaled = evaluate(model, test_loader, criterion)

predictions = ts.inverse_transform(pred_scaled).ravel()
actual = ts.inverse_transform(actual_scaled).ravel()

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))
r2 = r2_score(actual, predictions)

print(f"\nTEST RESULTS (5-min ahead, clean data)")
print(f"MSE (scaled): {test_loss:.6f}")
print(f"MAE:  {mae:.3f} mg/dL")
print(f"RMSE: {rmse:.3f} mg/dL")
print(f"R²:   {r2:.4f}")

# Save predictions CSV
results = pd.DataFrame({
    "actual_glucose": actual,
    "predicted_glucose": predictions,
    "absolute_error": np.abs(actual - predictions)
})
results.to_csv("test_predictions.csv", index=False)
print("Saved: test_predictions.csv")

# Training loss plot
plt.figure(figsize=(9,5))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.title('Bi-LSTM + Transformer Training (5-min horizon, clean data)')
plt.legend(); plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print("Saved: training_loss.png")

# Actual vs Predicted plot
n_plot = min(500, len(actual))
plt.figure(figsize=(12,5))
plt.plot(actual[:n_plot], label='Actual', linewidth=1.5)
plt.plot(predictions[:n_plot], label='Predicted', linewidth=1.5, linestyle='--')
plt.xlabel('Test Sample'); plt.ylabel('Glucose (mg/dL)')
plt.title(f'Actual vs Predicted (5-min ahead) | MAE={mae:.1f}, RMSE={rmse:.1f}')
plt.legend(); plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()
print("Saved: actual_vs_predicted.png")

print("\n✅ Model saved: bilstm_transformer_5min.pth")

In [ ]:
# ============================================================
# DOWNLOAD ALL OUTPUTS
# ============================================================
from google.colab import files
files.download('test_predictions.csv')
files.download('training_loss.png')
files.download('actual_vs_predicted.png')
files.download('bilstm_transformer_5min.pth')